In [ ]:
import pandas as pd

def network_merge(
    updated_base_node_df: pd.DataFrame,
    updated_base_link_df: pd.DataFrame,
    updated_merge_node_df: pd.DataFrame,
    updated_merge_link_df: pd.DataFrame,
    connector_df: pd.DataFrame,
    base_link_allowed_uses: str = None, # Changed default to None
    merge_link_allowed_uses: str = None  # Changed default to None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Merges updated base and merge network DataFrames, including connector links.
    Allows for optional overwriting of 'allowed_uses' for base and merge links.

    Args:
        updated_base_node_df (pd.DataFrame): Updated node DataFrame for the base network.
        updated_base_link_df (pd.DataFrame): Updated link DataFrame for the base network.
        updated_merge_node_df (pd.DataFrame): Updated node DataFrame for the network to be merged.
        updated_merge_link_df (pd.DataFrame): Updated link DataFrame for the network to be merged.
        connector_df (pd.DataFrame): DataFrame containing the generated connector links.
        base_link_allowed_uses (str, optional): String value to set for the 'allowed_uses' column
                                                of links from the base network. If None (default),
                                                the existing 'allowed_uses' values are kept.
        merge_link_allowed_uses (str, optional): String value to set for the 'allowed_uses' column
                                                 of links from the merged network. If None (default),
                                                 the existing 'allowed_uses' values are kept.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]:
            A tuple containing:
            - merged_node_df (pd.DataFrame): The combined node DataFrame.
            - merged_link_df (pd.DataFrame): The combined link DataFrame.
    """

    # 1. Merge Node DataFrames
    # Concatenate the updated base and merge node dataframes.
    # We use .copy() to ensure we're working with independent copies to avoid SettingWithCopyWarning
    merged_node_df = pd.concat(
    [updated_base_node_df.copy(), updated_merge_node_df.copy()],
    ignore_index=True,  # new index
    join='outer')

    # 2. Update 'allowed_uses' for base and merge links
    # Apply the specified allowed_uses string to the respective link dataframes if not None.
    if base_link_allowed_uses is not None:
        updated_base_link_df['allowed_uses'] = base_link_allowed_uses
    # Else, if base_link_allowed_uses is None, the original values are preserved.

    if merge_link_allowed_uses is not None:
        updated_merge_link_df['allowed_uses'] = merge_link_allowed_uses
    # Else, if merge_link_allowed_uses is None, the original values are preserved.


    # 3. Update link_id for connector links
    # Connector link_ids should start from (max_link_id from base/merge links) + 1.
    max_existing_link_id = max(updated_base_link_df['link_id'].max(), updated_merge_link_df['link_id'].max())

    # Only update connector_df if it's not empty, and it has 'link_id'
    if not connector_df.empty and 'link_id' in connector_df.columns:
        # Increment the existing connector link_ids by max_existing_link_id
        # This assumes connector_df's link_ids are already sequential from 1
        connector_df['link_id'] = connector_df['link_id'] + max_existing_link_id
    elif not connector_df.empty and 'link_id' not in connector_df.columns:
        print("Warning: 'link_id' column not found in connector_df. Connector link IDs will not be re-indexed.")
    
    # 4. Merge Link Dataframes
    # Concatenate the updated base links, updated merge links, and the adjusted connector links.
    merged_link_df = pd.concat(
    [
        updated_base_link_df.copy().reset_index(drop=True),
        updated_merge_link_df.copy().reset_index(drop=True),
        connector_df.copy().reset_index(drop=True)
    ],
    ignore_index=True,
    join='outer')

    merged_link_df = merged_link_df.sort_values(
        by=["from_node_id", "to_node_id"], 
        ascending=True
    ).reset_index(drop=True)

    merged_link_df['link_id'] = range(1, len(merged_link_df) + 1)

    return merged_node_df, merged_link_df


In [ ]:
#import os
#os.chdir("file path")

connector_df = pd.read_csv('transfer_connector_links.csv')

updated_base_node_df = pd.read_csv('updated_nodes.csv')
updated_merge_node_df = pd.read_csv('updated_bikeway_nodes.csv')
updated_base_link_df = pd.read_csv('updated_links.csv')
updated_merge_link_df = pd.read_csv('updated_bikeway_links.csv')

merged_nodes, merged_links = network_merge(
    updated_base_node_df=updated_base_node_df,
    updated_base_link_df=updated_base_link_df,
    updated_merge_node_df=updated_merge_node_df,
    updated_merge_link_df=updated_merge_link_df,
    connector_df=connector_df,
    base_link_allowed_uses="bike",
    merge_link_allowed_uses="bike" 
)

merged_nodes.to_csv('node_integrated.csv', index=False)
merged_links.to_csv('link_integrated.csv', index=False) 